# Neuron++ (giai đoạn 1: loss, text side, gộp mô tả)

Code: `zeroshot/Neuron/model/neuron_plus.py` (mô hình), `main_plus.py` (train / eval), config trong `config/plus/`.

Head thị giác giữ nguyên Neuron. Thêm (mỗi thứ bật tắt được để làm ablation):
- **loss**: `ce` (CE của Neuron, scale 0.1 cố định, hoặc scale học được), `xnce` (InfoCPL), `cal` (FS-VAE calibrated), `tri` (TDSM triplet margin); negative của `cal` / `tri` lấy từ đồ thị class (`neg_mode`: graph / random / all);
- **text side**: `adapter_rank` (adapter low-rank trên embedding mô tả), `prop_alpha` (lan truyền embedding qua đồ thị class);
- **gộp mô tả**: `agg` = mean (Neuron) / max / lse.

Mặc định (`RECIPE = 'base'`) = head và loss của Neuron. Kết quả mỗi mẫu **một** dự đoán: `zsl_s_neu` (spatial + temporal), `zsl_neu3` (cả 3 vòng), `zsl_neuron_or` (cách tính của paper Neuron, chỉ để đối chiếu). `zsl_prop_<mode>_a<alpha>`: cùng model, lan truyền embedding chỉ lúc test (không train lại; chọn alpha trên test là lạc quan). `graph` = đồ thị class; `self` (chỉ co về tâm chính class) và `random` (láng giềng ngẫu nhiên) là đối chứng: chỉ giữ đồ thị nếu nó thắng cả hai. GZSL: γ dò trên test như Neuron.


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, subprocess
import pandas as pd

DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'
SKEL = {'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
        'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz'}
OUT_DIR = f'{DRIVE_DIR}/neuron_plus'          # work_dir trong config/plus/*.yaml: <OUT_DIR>/<recipe>/<split>

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
NEURON = f'{REPO}/zeroshot/Neuron'
print('weights:', len(glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True)), 'file')


In [ ]:
# Giải nén npz trên Drive thẳng thành .npy trên đĩa local (đọc tuần tự, không nạp vào RAM).
# Feeder_Mmap đọc các .npy này bằng memmap, nên chạy được cả trên T4 (RAM ~12 GB).
import zipfile
def local_npy(ds):
    out = f"/content/data/{os.path.basename(SKEL[ds])[:-4]}"
    os.makedirs(out, exist_ok=True)
    with zipfile.ZipFile(SKEL[ds]) as zf:
        for name in zf.namelist():
            dst = f"{out}/{name}"
            if not os.path.exists(dst):
                with zf.open(name) as s, open(dst + ".tmp", "wb") as d:
                    shutil.copyfileobj(s, d, 64 << 20)
                os.replace(dst + ".tmp", dst)
    print(out, sorted(os.listdir(out)))

local_npy('ntu60')
!free -h | head -2; df -h /content | tail -1


In [ ]:
# Chạy một split với một công thức (recipe). Lần đầu: trích cube của encoder một lần (~5-11 phút);
# các recipe sau dùng lại cube trong /content/cube_cache. Bị ngắt thì chạy lại cell này: tiếp tục từ last.pt.
# Thứ tự ablation gợi ý: base -> scale -> (xnce | cal_graph | cal_random | tri_graph) -> lse -> adapter -> prop.
# So cal_graph với cal_random để biết negative từ đồ thị có ích hơn negative ngẫu nhiên không.
CFG    = 'ntu60_xsub_48_12'   # ntu60_xsub_55_5 | ntu60_xsub_48_12 | ntu120_xsub_110_10 | ntu120_xsub_96_24
RECIPE = 'base'
SC = 'ce_scale=10,ce_scale_learn=True'      # CE kiểu CLIP: scale học được, khởi tạo 10 (Neuron: 0.1 cố định)
RECIPES = {
    'base':             '',
    'scale':            f'--model-args {SC}',
    'scale_xnce':       f'--lambda-xnce 1 --model-args {SC}',
    'scale_cal_graph':  f'--lambda-cal 1 --model-args {SC},neg_mode=graph',
    'scale_cal_random': f'--lambda-cal 1 --model-args {SC},neg_mode=random',
    'scale_tri_graph':  f'--lambda-tri 1 --model-args {SC},neg_mode=graph',
    'scale_lse':        f'--model-args {SC},agg=lse',
    'scale_adapter':    f'--model-args {SC},adapter_rank=32',
    'scale_prop':       f'--model-args {SC},prop_alpha=0.2',
    # PartAttn: new head (model/part_attn.py) on the same frozen encoder; config/part/ has its own optim + loss defaults
    'part':             f'--config config/part/{CFG}.yaml',
    'part_lse':         f'--config config/part/{CFG}.yaml --model-args agg=lse',
    'part_ce_only':     f'--config config/part/{CFG}.yaml --lambda-xnce 0',
    'part_cal_graph':   f'--config config/part/{CFG}.yaml --lambda-cal 1',
}
EXTRA  = RECIPES[RECIPE]    # hoặc tự ghi đè, vd '--lambda-cal 0.5 --seed 2'
SPLIT = {'ntu60_xsub_55_5': 'ntu60_seen55_unseen5', 'ntu60_xsub_48_12': 'ntu60_seen48_unseen12',
         'ntu120_xsub_110_10': 'ntu120_seen110_unseen10', 'ntu120_xsub_96_24': 'ntu120_seen96_unseen24'}[CFG]
if CFG.startswith('ntu120'):
    local_npy('ntu120')
!cd {NEURON} && python main_plus.py --config config/plus/{CFG}.yaml --work-dir {OUT_DIR}/{RECIPE}/{SPLIT} {EXTRA}


In [ ]:
# Tổng hợp mọi run đã xong: <OUT_DIR>/<recipe>/<split>/summary.json
rows = []
for f in sorted(glob.glob(f'{OUT_DIR}/*/*/summary.json')):
    s = json.load(open(f))
    run = os.path.basename(os.path.dirname(os.path.dirname(f)))
    last = s['last_epoch']
    tail = pd.read_csv(os.path.join(os.path.dirname(f), 'metrics.csv')).tail(10)   # trung bình 10 epoch cuối
    props = [k for k in last if k.startswith('zsl_prop_') and k.endswith(('_a0.5', '_a0.9'))]
    for name, r in (('last', last), ('mean last 10', tail.mean().to_dict()),
                    (f"best.pt (ep {s['best_epoch']})", s['best_checkpoint'])):
        rows.append(dict(run=run, split=s['split'], row=name,
                         **{k: r.get(k) for k in ['zsl_s_neu', 'zsl_neu3', 'zsl_neuron_or', 'gzsl_s_neu_H'] + props}))
display(pd.DataFrame(rows).round(4))


In [ ]:
# Đường học theo epoch của CFG / RECIPE (đọc metrics.csv, xem được cả khi đang train) và accuracy từng class unseen
d = f'{OUT_DIR}/{RECIPE}/{SPLIT}'
path = f'{d}/metrics.csv'
if not os.path.exists(path):
    print('chưa có epoch nào xong:', path)
else:
    m = pd.read_csv(path)
    cols = ['zsl_s_neu', 'zsl_neu3', 'zsl_neuron_or'] + [c for c in m if c.startswith('zsl_prop_') and c.endswith('_a0.5')]
    m.plot(x='epoch', y=cols, figsize=(9, 4), grid=True, title=f'ZSL theo epoch ({RECIPE} / {SPLIT})')
    show = ['epoch'] + cols + [c for c in ('gzsl_s_neu_H', 'ce_scale', 'time_s') if c in m] + [c for c in m if c.startswith('L_')]
    display(m[show].round(4))
    pc = f'{d}/per_class_best.json'
    if os.path.exists(pc):
        print(json.load(open(pc)))
